In [1]:
%pip install youtube-transcript-api

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import time
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable
)

In [8]:
df = pd.read_csv("cleaned_metadata.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (250, 6)


,video_id,title,publish_date,year,month,title_length
0,1Z-8HxlLAHk,Kids and young people: stay curious and be wil...,2026-03-30 12:17:27+00:00,2026,3,72
1,ehkDsePv3W8,Here's a cool and easy way to work with colors...,2026-03-29 12:43:24+00:00,2026,3,72
2,uWRdzJTpcpI,Do web devs NEED to understand low-level progr...,2026-03-28 12:32:40+00:00,2026,3,88
3,GC5CLCgnvm0,"When things are new and a little scary, embrac...",2026-03-27 12:18:22+00:00,2026,3,79
4,tZef2ZzbyuQ,What happens when the model CAN'T fix it? Inte...,2026-03-27 10:00:57+00:00,2026,3,99


In [9]:
if "video_id" in df.columns:
    print("video_id column exists ✅")
else:
    print("video_id column missing ❌")

video_id column exists ✅


In [10]:
df["video_id"] = df["video_id"].astype(str).str.strip()

In [11]:
video_id = df["video_id"].iloc[0]

api = YouTubeTranscriptApi()

try:
    transcript = api.fetch(video_id, languages=["en","hi"])

    print(transcript[0].text)

except Exception as e:
    print("Transcript not available:", e)

like and even at a young age like


In [12]:
transcript_text = " ".join([seg.text for seg in transcript])

print(transcript_text[:500])

like and even at a young age like learning to engage with people, learning to ask questions, right? [snorts] Be curious, be inquisitive, right? Um I mean, teaching your kid to be inquisitive, to be curious, to ask follow-up questions, to engage with people. I mean, that's how a a younger person can get some of those kinds of opportunities that that come along. So, it's just kind of keeping your ears open and being willing to engage. As broad and vague as that sounds, that would be the biggest pi


In [13]:
api = YouTubeTranscriptApi()

def get_transcript(video_id):

    try:
        transcript = api.fetch(video_id, languages=["en","hi"])

        text = " ".join([seg.text for seg in transcript])

        return text

    except Exception:
        return None

In [14]:
transcripts = []
failures = []

print("Extracting transcripts...\n")

for vid in df["video_id"]:

    text = get_transcript(vid)

    transcripts.append(text)

    if text is None:
        failures.append({
            "video_id": vid,
            "reason": "Transcript unavailable"
        })

    time.sleep(0.2)   

Extracting transcripts...



In [15]:
df["transcript"] = transcripts

df.tail()

,video_id,title,publish_date,year,month,title_length,transcript
245,Ufy1FyqvfKM,Audit the advice you're getting - it's not all...,2025-10-17 23:32:25+00:00,2025,10,60,"On the program podcast, we recently covered th..."
246,hM2T45qbIPI,From injured athlete to software engineer with...,2025-10-17 14:00:06+00:00,2025,10,74,Welcome back to the Free Code Camp podcast. I'...
247,_dCn4WrWnBw,How to use Python's .isprintable() method,2025-10-16 23:18:24+00:00,2025,10,41,To check if all characters in a string are pri...
248,c6IyCwAV6BY,What is the JavaScript DOM?,2025-10-16 17:58:03+00:00,2025,10,27,Learn all about the document object model or D...
249,Y9m_Chx1joE,Why you should surround yourself with the smar...,2025-10-15 22:52:59+00:00,2025,10,70,You go find the smartest people because they a...


In [16]:
total = len(df)

available = df["transcript"].notnull().sum()

missing = df["transcript"].isnull().sum()

print("Total videos:", total)
print("Transcripts available:", available)
print("Missing transcripts:", missing)

coverage = (available / total) * 100

print("Transcript coverage:", round(coverage,2), "%")

Total videos: 250
Transcripts available: 244
Missing transcripts: 6
Transcript coverage: 97.6 %


In [17]:
fail_df = pd.DataFrame(failures)

fail_df.to_csv("transcript_failures.csv", index=False)

print("Failure log saved")

Failure log saved


In [18]:
df.to_csv("video_with_transcripts.csv", index=False)

print("Dataset saved as video_with_transcripts.csv")

Dataset saved as video_with_transcripts.csv
